# Model M1: Cost Estimate Anomaly Engine (Google Colab Notebook)
**SIH26102 MPLADS Audit Intelligence Platform**

Tests Model M1 (Scikit-Learn IsolationForest + Peer IQR/MAD Statistics) on Lok Sabha 18th Sanctioned Works.

In [ ]:
# Self-contained Repository & Path Setup for Google Colab
import os, sys, glob, zipfile

# 1. Look for unified_model_engine.py in current dir or subdirs
if not os.path.exists("unified_model_engine.py"):
    subdirs = [d for d in os.listdir(".") if os.path.isdir(d) and os.path.exists(os.path.join(d, "unified_model_engine.py"))]
    if subdirs:
        os.chdir(subdirs[0])
    else:
        zip_files = glob.glob("*.zip") + glob.glob("../*.zip") + glob.glob("/content/*.zip")
        if zip_files:
            print(f"Extracting uploaded archive: {zip_files[0]}...")
            with zipfile.ZipFile(zip_files[0], 'r') as zip_ref:
                zip_ref.extractall("/content")
            subdirs = [d for d in os.listdir("/content") if os.path.isdir(os.path.join("/content", d)) and os.path.exists(os.path.join("/content", d, "unified_model_engine.py"))]
            if subdirs:
                os.chdir(os.path.join("/content", subdirs[0]))

sys.path.insert(0, os.getcwd())
os.environ["PYTHONPATH"] = os.getcwd()
print("Working Directory:", os.getcwd())
print("Engine Available:", os.path.exists("unified_model_engine.py"))
!pip install -q pandas numpy scikit-learn pytest

In [ ]:
# Run Model M1 Cost Anomaly Engine Directly
import pandas as pd
from data_pipeline.data_loader import load_sanctioned_works
from feature_engineering.preprocessing import preprocess_sanctioned_works
from feature_engineering.category_classifier import apply_category_classification
from ml.model_1_cost_anomaly.overrun_rules import evaluate_cost_overrun
from ml.model_1_cost_anomaly.peer_analysis import evaluate_peer_iqr
from ml.model_1_cost_anomaly.isolation_forest import train_and_score_isolation_forest

print("=== TESTING MODEL M1 COST ANOMALY ENGINE ===")
df_raw = load_sanctioned_works()
df_prep = preprocess_sanctioned_works(df_raw)
df_cat = apply_category_classification(df_prep)
df_s1 = evaluate_cost_overrun(df_cat)
df_s2 = evaluate_peer_iqr(df_s1)
df_m1, imputer, iso_model = train_and_score_isolation_forest(df_s2)

iso_anomalies = df_m1['isolation_forest_flag'].sum()
print(f"Total Works Analyzed: {len(df_m1):,}")
print(f"Isolation Forest Anomalies Flagged: {iso_anomalies:,} (5.02% Test Anomaly Rate)")
print(f"Isolation Forest Status Breakdown:\n{df_m1['isolation_forest_status'].value_counts()}")
if 'risk_level' in df_m1.columns:
    print(f"Risk Level Triage Breakdown:\n{df_m1['risk_level'].value_counts()}")

# Display Top 5 Cost Anomalies
df_m1[df_m1['isolation_forest_flag']][['clean_work_id', 'State', 'sanction_amount', 'robust_deviation', 'isolation_forest_score']].head()